# Next-Latent Prediction: Autoregression in Latent Space

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/llm/next_latent_prediction.ipynb)

A next-token model predicts a discrete symbol from a softmax. A **next-latent** model predicts
the next *continuous* vector autoregressively and only decodes to tokens at the surface. This
notebook builds a tiny version from scratch to show the one thing that makes it hard: with a
continuous target there is no cross-entropy to lean on, so if the next latent is multimodal a
regression (MSE) head **collapses to the mean** of the plausible continuations, a point that
decodes to nothing. A codebook (VQ) head or a diffusion head fixes it.

Everything runs on CPU in a couple of minutes. Companion post: *Next-Latent Prediction* on sesen.ai.

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
DEVICE = "cpu"
torch.manual_seed(0)

## 1. A corpus worth compressing

A dictionary of `M` "words", each a fixed template of `K` tokens over a `V`-symbol vocabulary.
Sentences are word sequences from an order-1 Markov chain whose transitions are spread over a
few near-equiprobable successors, so the next word is uncertain and the head choice
will matter.

In [ ]:
def make_words(M, K, V, seed=0):
    rng = np.random.default_rng(seed); words, seen = [], set()
    while len(words) < M:
        w = tuple(rng.integers(0, V, K).tolist())
        if w not in seen: seen.add(w); words.append(w)
    return np.array(words)

def make_transition(M, fanout=3, seed=1):
    rng = np.random.default_rng(seed); T = np.zeros((M, M))
    for i in range(M):
        succ = rng.choice(M, size=fanout, replace=False)
        p = rng.uniform(0.8, 1.2, fanout); T[i, succ] = p / p.sum()
    return T

def true_next_entropy(T):
    pi = np.full(T.shape[0], 1 / T.shape[0])
    for _ in range(2000): pi = pi @ T
    rowH = -np.sum(np.where(T > 0, T * np.log(T + 1e-12), 0.0), 1)
    return float(pi @ rowH)

def sample_corpus(n_seq, W, words, T, seed=2):
    rng = np.random.default_rng(seed); M, K = words.shape
    wid = np.zeros((n_seq, W), dtype=np.int64); wid[:, 0] = rng.integers(0, M, n_seq)
    cdf = np.cumsum(T, 1)
    for t in range(1, W):
        u = rng.random(n_seq)[:, None]; wid[:, t] = (cdf[wid[:, t - 1]] > u).argmax(1)
    return wid, words[wid].reshape(n_seq, W * K).astype(np.int64)

V, M, K, W, d = 16, 8, 4, 16, 16
words = make_words(M, K, V); T = make_transition(M)
H_true = true_next_entropy(T)
wid_tr, tok_tr = sample_corpus(2500, W, words, T, seed=2)
wid_va, tok_va = sample_corpus(500, W, words, T, seed=3)
print(f"token stream = {W*K}, latent stream = {W}  ({K}x shorter)")
print(f"true H(next word | word) = {H_true:.3f} nats  (perplexity {np.exp(H_true):.2f} of {M})")

## 2. A word autoencoder: K tokens <-> one latent

Once trained, this reconstructs every word perfectly, so the latents are lossless. Its per-word
latents become the codebook.

In [ ]:
class WordAutoencoder(nn.Module):
    def __init__(self, V, K, d, h=64):
        super().__init__(); self.V, self.K = V, K
        self.embed = nn.Embedding(V, 16)
        self.enc = nn.Sequential(nn.Linear(K * 16, h), nn.GELU(), nn.Linear(h, d))
        self.dec = nn.Sequential(nn.Linear(d, h), nn.GELU(), nn.Linear(h, K * V))
    def encode(self, tok): return self.enc(self.embed(tok).flatten(1))
    def decode(self, z):   return self.dec(z).view(-1, self.K, self.V)

ae = WordAutoencoder(V, K, d)
opt = torch.optim.Adam(ae.parameters(), 3e-3)
x = torch.as_tensor(words)
for _ in range(400):
    opt.zero_grad()
    loss = F.cross_entropy(ae.decode(ae.encode(x)).reshape(-1, V), x.reshape(-1))
    loss.backward(); opt.step()
for p in ae.parameters(): p.requires_grad_(False)
with torch.no_grad():
    acc = (ae.decode(ae.encode(x)).argmax(-1) == x).float().mean().item()
    codebook = ae.encode(x)                                   # (M, d) frozen codes
    lat_tr = ae.encode(torch.as_tensor(tok_tr).view(-1, K)).view(len(tok_tr), W, d).numpy()
    lat_va = ae.encode(torch.as_tensor(tok_va).view(-1, K)).view(len(tok_va), W, d).numpy()
print(f"autoencoder token reconstruction accuracy = {acc*100:.1f}%")

## 3. Autoregression over the latent stream, three heads

A small causal Transformer predicts the next latent. **mse** regresses it (collapses); **vq**
classifies the next word over the codebook (a softmax reappears); **diffusion** denoises the
next latent conditioned on context.

In [ ]:
def ddpm(steps=50):
    beta = torch.linspace(1e-3, 0.15, steps); abar = torch.cumprod(1 - beta, 0)
    return beta, abar

class LatentAR(nn.Module):
    def __init__(self, d, M, head, dm=64, L=2, hds=4, maxlen=64):
        super().__init__(); self.head, self.d, self.M = head, d, M
        self.inp = nn.Linear(d, dm); self.pos = nn.Parameter(torch.zeros(1, maxlen, dm))
        enc = nn.TransformerEncoderLayer(dm, hds, dm * 2, batch_first=True,
                                         norm_first=True, activation="gelu", dropout=0.0)
        self.net = nn.TransformerEncoder(enc, L)
        if head == "vq":  self.out = nn.Linear(dm, M)
        elif head == "mse": self.out = nn.Linear(dm, d)
        else:
            self.ctx = nn.Linear(dm, dm)
            self.den = nn.Sequential(nn.Linear(d + dm + 1, 128), nn.GELU(),
                                     nn.Linear(128, 128), nn.GELU(), nn.Linear(128, d))
    def context(self, z):
        h = self.inp(z) + self.pos[:, :z.size(1)]
        m = torch.triu(torch.ones(z.size(1), z.size(1)), 1).bool()
        return self.net(h, mask=m)

def train_latent(m, lat, wid, epochs=300, steps=50):
    opt = torch.optim.Adam(m.parameters(), 2e-3)
    z = torch.as_tensor(lat, dtype=torch.float32); ids = torch.as_tensor(wid)
    beta, abar = ddpm(steps)
    for _ in range(epochs):
        opt.zero_grad(); h = m.context(z[:, :-1]); tz, tid = z[:, 1:], ids[:, 1:]
        if m.head == "vq":
            loss = F.cross_entropy(m.out(h).reshape(-1, m.M), tid.reshape(-1))
        elif m.head == "mse":
            loss = F.mse_loss(m.out(h), tz)
        else:
            c = m.ctx(h); B, Wn, D = tz.shape; t = torch.randint(0, steps, (B, Wn))
            ab = abar[t].unsqueeze(-1); noise = torch.randn_like(tz)
            zt = ab.sqrt() * tz + (1 - ab).sqrt() * noise
            loss = F.mse_loss(m.den(torch.cat([zt, c, t.unsqueeze(-1).float() / steps], -1)), noise)
        loss.backward(); opt.step()
    return float(loss.detach())

@torch.no_grad()
def diffusion_sample(m, hc, steps=50):
    beta, abar = ddpm(steps); c = m.ctx(hc); x = torch.randn(hc.size(0), m.d)
    for t in reversed(range(steps)):
        tt = torch.full((hc.size(0), 1), t / steps)
        eps = m.den(torch.cat([x, c, tt], -1))
        x = (x - beta[t] / (1 - abar[t]).sqrt() * eps) / (1 - beta[t]).sqrt()
        if t > 0: x = x + beta[t].sqrt() * torch.randn_like(x)
    return x

heads = {}
for h in ["vq", "mse", "diffusion"]:
    m = LatentAR(d, M, h); train_latent(m, lat_tr, wid_tr, epochs=400); heads[h] = m; print("trained", h)

## 4. The head decides everything

Two robust diagnostics. **Distance to nearest code** asks whether the predicted latent lands on a
real word: `0` means exactly on a code, larger means out in the void between codes. **Conditional
entropy** asks whether the model reproduces the branching: the true chain has `H = 1.09` nats, and a
collapsed head that always answers the same way scores `0`.

In [ ]:
@torch.no_grad()
def sample_next_words(m, ctx, n=400, seed=0):
    torch.manual_seed(seed); wt = torch.as_tensor(words)
    z = torch.as_tensor(ctx, dtype=torch.float32).view(-1, 1, d).repeat_interleave(n, 0)
    h = m.context(z)[:, -1]
    if m.head == "vq":
        nxt = codebook[torch.multinomial(F.softmax(m.out(h), -1), 1).squeeze(1)]
    elif m.head == "diffusion":
        nxt = diffusion_sample(m, h)
    else:
        nxt = m.out(h)
    toks = ae.decode(nxt).argmax(-1)
    match = (toks.unsqueeze(1) == wt.unsqueeze(0)).all(-1)
    ids = torch.where(match.any(1), match.float().argmax(1), torch.full((z.size(0),), -1.0)).long()
    return ids.view(-1, n).numpy()

def entropy(ids, M):
    c = np.zeros(M + 1)
    for w in ids: c[w if w >= 0 else M] += 1
    p = c / c.sum(); return float(-np.sum(np.where(p > 0, p * np.log(p), 0.0)))

@torch.no_grad()
def code_distance(m, lat):                            # min distance to any code / mean code spacing
    z = torch.as_tensor(lat, dtype=torch.float32); h = m.context(z[:, :-1])
    if m.head == "mse":
        pred = m.out(h).reshape(-1, d)
    elif m.head == "diffusion":
        pred = diffusion_sample(m, h.reshape(-1, h.size(-1)))
    else:
        pred = codebook[m.out(h).reshape(-1, M).argmax(-1)]
    dmin = torch.cdist(pred, codebook).min(1).values
    return (dmin / torch.pdist(codebook).mean()).mean().item()

cb = codebook.numpy()
print(f"{'head':<11}{'dist to code':>14}{'diversity (nats)':>18}")
for h in ["vq", "diffusion", "mse"]:
    dist = code_distance(heads[h], lat_va)
    div = np.mean([entropy(sample_next_words(heads[h], cb, n=400)[i], M) for i in range(M)])
    print(f"{h:<11}{dist:>14.2f}{div:>18.2f}")
print(f"{'(true)':<11}{'':>14}{H_true:>18.2f}")

### Latent space: where predictions land

VQ predictions land on the real word codes; MSE predictions fall into the empty space between
them (the mean of the plausible successors).

In [ ]:
@torch.no_grad()
def predict(m, ctx, seed=1):
    torch.manual_seed(seed); z = torch.as_tensor(ctx, dtype=torch.float32).unsqueeze(1)
    h = m.context(z)[:, -1]
    if m.head == "vq":
        return codebook[torch.multinomial(F.softmax(m.out(h), -1), 1).squeeze(1)].numpy()
    return m.out(h).numpy()

vq_p, mse_p = predict(heads["vq"], cb), predict(heads["mse"], cb)
mean = cb.mean(0); _, _, Vt = np.linalg.svd(np.concatenate([cb, vq_p, mse_p]) - mean, full_matrices=False)
P = lambda X: (X - mean) @ Vt[:2].T
cb2, vq2, mse2 = P(cb), P(vq_p), P(mse_p)
plt.figure(figsize=(7, 6))
plt.scatter(*cb2.T, s=320, marker="*", color="#7c3aed", ec="white", label="word codes", zorder=4)
plt.scatter(*vq2.T, s=80, color="#2563eb", ec="white", label="VQ predictions", zorder=5)
plt.scatter(*mse2.T, s=100, marker="X", color="#dc2626", ec="white", label="MSE predictions", zorder=6)
for i in range(M): plt.annotate(f"w{i}", cb2[i], xytext=(0, 12), textcoords="offset points", ha="center")
plt.legend(); plt.xticks([]); plt.yticks([])
plt.title("VQ lands on codes; MSE falls into the void between them"); plt.show()

### Next-word distribution after one word

The true chain branches to three successors. VQ matches it, diffusion approximates it, MSE spikes
on a single mode.

In [ ]:
src = int(np.argmax([np.count_nonzero(T[i]) for i in range(M)]))
dists = {}
for h in ["vq", "mse", "diffusion"]:
    ids = sample_next_words(heads[h], cb[[src]], n=2000, seed=2)[0]
    c = np.zeros(M + 1)
    for w in ids: c[w if w >= 0 else M] += 1
    dists[h] = c / c.sum()
x = np.arange(M + 1); labels = [f"w{i}" for i in range(M)] + ["inv"]
plt.figure(figsize=(9, 4))
plt.bar(x - 0.3, np.append(T[src], 0), 0.2, color="#111", label="true")
plt.bar(x - 0.1, dists["vq"], 0.2, color="#2563eb", label="VQ")
plt.bar(x + 0.1, dists["diffusion"], 0.2, color="#16a34a", label="diffusion")
plt.bar(x + 0.3, dists["mse"], 0.2, color="#dc2626", label="MSE")
plt.xticks(x, labels); plt.ylabel("P(next word)"); plt.legend()
plt.title(f"Next-word distribution after word w{src}"); plt.show()

## Exercises

1. **Fanout.** Raise `fanout` in `make_transition` so each word has more successors. Does the MSE
   valid-word rate fall further as the mean drifts deeper into the void?
2. **Codebook size.** Replace the per-word codebook with a learned codebook of size other than `M`.
   How small can it get before VQ starts to blur words together?
3. **Diffusion steps.** Increase the diffusion sampling steps. Does the valid-word rate climb toward
   the VQ head's, and at what cost?
4. **Mixture head.** Add a head that predicts a small Gaussian mixture over the next latent (means +
   weights). Does it recover the branching without a codebook?
5. **Longer memory.** Make the chain order-2 (next word depends on the previous two). Does the
   Transformer latent model still match the true entropy?

### References

- Hao et al. (2024), *Training Large Language Models to Reason in a Continuous Latent Space* (Coconut)
- Assran et al. (2023), *Self-Supervised Learning from Images with a Joint-Embedding Predictive Architecture* (I-JEPA)
- LeCun (2022), *A Path Towards Autonomous Machine Intelligence*
- van den Oord et al. (2017), *Neural Discrete Representation Learning* (VQ-VAE)